In [43]:
import warnings
import tracemalloc

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.preprocessing import (
    PowerTransformer, QuantileTransformer, MinMaxScaler, 
    StandardScaler, RobustScaler, KBinsDiscretizer
)
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.datasets import load_iris

from imblearn import FunctionSampler
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from feature_engine.outliers import Winsorizer, OutlierTrimmer

#warnings.filterwarnings('ignore')

In [44]:
def run_experiments(X_list, y_list, dataset_names):
    models = {
        'Gaussian Naive Bayes': GaussianNB(),
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=2000),
        'KNN': KNeighborsClassifier(),
        'SVM': SVC(random_state=42, max_iter=2000),
        'Random Forest': RandomForestClassifier(random_state=42),
        'MLP': MLPClassifier(random_state=42, max_iter=2000)
    }

    def Trimmer(X, y, capping_method, fold, variables):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
            
        trimmer = OutlierTrimmer(capping_method=capping_method, tail='both', fold=fold, variables=variables)
        X_res = trimmer.fit_transform(X)
        
        y_was_array = isinstance(y, np.ndarray)
        if y_was_array:
            y = pd.Series(y, index=X.index)
            
        y_res = y.loc[X_res.index]
        
        if y_was_array:
            y_res = y_res.values
            
        return X_res, y_res

    metrics = [
        'balanced_accuracy', 
        'precision_macro', 
        'recall_macro', 
        'f1_macro', 
        'average_precision'
    ]
    
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    for X, y, name in zip(X_list, y_list, dataset_names):
        
        continuous_cols = [col for col in X.columns if not col.startswith('categorical_')]
        
        preprocessors = {
            'Baseline': 'passthrough',
            #'IQR Capping': Winsorizer(capping_method='iqr', tail='both', fold=1.5, variables=continuous_cols),
            #'Quantile Capping': Winsorizer(capping_method='quantiles', tail='both', fold=0.05, variables=continuous_cols),
            #'IQR Removal': FunctionSampler(func=Trimmer, kw_args={'capping_method': 'iqr', 'fold': 1.5, 'variables': continuous_cols}, validate=False),
            #'Quantile Removal': FunctionSampler(func=Trimmer, kw_args={'capping_method': 'mad', 'fold': 0.5, 'variables': continuous_cols}, validate=False),
            'Yeo-Johnson': PowerTransformer(method='yeo-johnson'),
            'Quantile Transform': QuantileTransformer(output_distribution='normal', random_state=42),
            'Min-Max Scaler': MinMaxScaler(),
            'Standard Scaler': StandardScaler(),
            'Robust Scaler': RobustScaler(),
            'Uniform Binning': KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform', subsample=None),
            'Quantile Binning': KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile', subsample=None),
            'Random Undersampling': RandomUnderSampler(random_state=42),
            'SMOTE': SMOTE(random_state=42)
        }
        
        pbar = tqdm(total=len(models) * len(preprocessors), desc=f"Evaluating: {name}")

        for model_name, model in models.items():
            for prep_name, prep in preprocessors.items():
                
                steps = []
                
                if prep == 'passthrough':
                    steps.append(('preprocessor', 'passthrough'))
                elif isinstance(prep, (SMOTE, RandomUnderSampler, FunctionSampler)):
                    steps.append(('sampler', prep))
                else:
                    ct = ColumnTransformer(
                        transformers=[('num', prep, continuous_cols)],
                        remainder='passthrough'
                    )
                    steps.append(('preprocessor', ct))
                
                steps.append(('classifier', model))
                pipeline = Pipeline(steps)
                
                tracemalloc.start()
                
                try:
                    scores = cross_validate(
                        pipeline, X, y, 
                        cv=cv, scoring=metrics, n_jobs=-1, return_train_score=False
                    )
                    
                    _, peak = tracemalloc.get_traced_memory()
                    tracemalloc.stop()
                    
                    results.append({
                        'Dataset': name,
                        'Model': model_name,
                        'Preprocessor': prep_name,
                        'Fit Time (s)': np.mean(scores['fit_time']),
                        'Predict Time (s)': np.mean(scores['score_time']),
                        'Peak Memory (MB)': peak / (1024 * 1024),
                        'Balanced Accuracy': np.mean(scores['test_balanced_accuracy']),
                        'Precision Macro': np.mean(scores['test_precision_macro']),
                        'Recall Macro': np.mean(scores['test_recall_macro']),
                        'F1 Macro': np.mean(scores['test_f1_macro']),
                        'Average Precision': np.mean(scores['test_average_precision']),
                        'Error': None
                    })
                    
                except Exception as e:
                    tracemalloc.stop()
                    results.append({
                        'Dataset': name,
                        'Model': model_name,
                        'Preprocessor': prep_name,
                        'Fit Time (s)': np.nan,
                        'Predict Time (s)': np.nan,
                        'Peak Memory (MB)': np.nan,
                        'Balanced Accuracy': np.nan,
                        'Precision Macro': np.nan,
                        'Recall Macro': np.nan,
                        'F1 Macro': np.nan,
                        'Average Precision': np.nan,
                        'Error': str(e)
                    })
                
                pbar.update(1)
        
        pbar.close()
        
    return pd.DataFrame(results)

In [45]:
df_iranian = pd.read_csv('data/Customer Churn.csv', sep=',')
categorical_cols_iranian = ['Complains', 'Status', 'Tariff Plan']
df_iranian = pd.get_dummies(df_iranian, columns=categorical_cols_iranian, drop_first=True, dtype=int, prefix=['categorical_' + col for col in categorical_cols_iranian], prefix_sep='_')
y_iranian = df_iranian['Churn']
X_iranian = df_iranian.drop(columns=['Churn'])

df_credit = pd.read_excel('data/default of credit card clients.xls', header=1)
categorical_cols_credit = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
df_credit = pd.get_dummies(df_credit, columns=categorical_cols_credit, drop_first=True, dtype=int,  prefix=['categorical_' + col for col in categorical_cols_credit], prefix_sep='_')
y_credit = df_credit['default payment next month'].rename('y')
X_credit = df_credit.drop(columns=['ID', 'default payment next month'])

df_shoppers = pd.read_csv('data/online_shoppers_intention.csv')
categorical_cols_shoppers = ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']
df_shoppers = pd.get_dummies(df_shoppers, columns=categorical_cols_shoppers, drop_first=True, dtype=int,  prefix=['categorical_' + col for col in categorical_cols_shoppers], prefix_sep='_')
y_shoppers = df_shoppers['Revenue'].astype(int).rename('y')
X_shoppers = df_shoppers.drop(columns=['Revenue'])

X = [X_iranian, X_credit, X_shoppers]
y = [y_iranian, y_credit, y_shoppers]
dataset_names = ['iranian', 'credit', 'shoppers']

In [ ]:
df_final = run_experiments(X, y, dataset_names)
df_final.to_csv('data/experiment_results.csv', index=False)
df_final

Evaluating: iranian:   0%|          | 0/60 [00:00<?, ?it/s]

Evaluating: credit:   0%|          | 0/60 [00:00<?, ?it/s]

Evaluating: shoppers:   0%|          | 0/60 [00:00<?, ?it/s]

,Dataset,Model,Preprocessor,Fit Time (s),Predict Time (s),Peak Memory (MB),Balanced Accuracy,Precision Macro,Recall Macro,F1 Macro,Average Precision,Error
0,iranian,Gaussian Naive Bayes,Baseline,0.005526,0.015331,103.427452,0.811487,0.672639,0.811487,0.668159,0.683011,None
1,iranian,Gaussian Naive Bayes,Yeo-Johnson,0.048055,0.016880,1.296936,0.841566,0.718540,0.841566,0.745311,0.728085,None
2,iranian,Gaussian Naive Bayes,Quantile Transform,0.075372,0.024869,1.294377,0.821550,0.733240,0.821550,0.760655,0.703299,None
3,iranian,Gaussian Naive Bayes,Min-Max Scaler,0.006782,0.015137,1.294286,0.814067,0.675057,0.814067,0.673017,0.676340,None
4,iranian,Gaussian Naive Bayes,Standard Scaler,0.006473,0.014728,1.152514,0.814067,0.675057,0.814067,0.673017,0.677014,None
...,...,...,...,...,...,...,...,...,...,...,...,...
175,shoppers,MLP,Robust Scaler,5.901638,0.020525,9.370135,0.769114,0.808526,0.769114,0.785829,0.686141,None
176,shoppers,MLP,Uniform Binning,32.044788,0.020474,9.370290,0.569327,0.605645,0.569327,0.578549,0.307913,None
177,shoppers,MLP,Quantile Binning,33.118290,0.020071,9.370424,0.740617,0.752955,0.740617,0.746285,0.604468,None
178,shoppers,MLP,Random Undersampling,0.445826,0.014751,9.366138,0.784467,0.700753,0.784467,0.698474,0.540847,None


# Comentários

- Erro nas técnicas de outliers devido a Q1 = Q3
- Os artigos que utilizam os testes escolhidos sugerem um mínimo de 30 datasets
- Somente dados reais?
- Somente dados desbalanceados?
- Numero min e max de linhas?
- Numero min e max de colunas?